# S6E8: mix the meta-models, then fix the bands where the blend is weakest

**August 13 update:** the final section adds a deliberately simple rank blend of this
notebook's audited 94-member output, Naji's 0.97097 stack and Nikita's independent model.
The exact output has now scored **0.97099 public LB**. This last step is leaderboard-informed,
so it is kept separate from the OOF experiments below rather than passed off as another
cross-validation result.

The modelling work in the main notebook reaches out-of-fold ROC AUC **0.969721**, averaged
over three independent meta-CV partitions, against **0.969694** for the strongest published
blend of the same members. The gain is positive in **15 of 15** folds.

Three things are new here.

**1. A factorization machine over the value lattice.** The public
[OOF library](https://www.kaggle.com/datasets/szymonkapiski/s6e8-oof-library-47-models) holds
74 models: 11 XGBoost, 19 LightGBM, 12 CatBoost, 9 TabM, 5 RealMLP, 1 transformer, 1 TabNet
and a handful of MLPs. There is no bilinear model in it, and this dataset is unusually well
suited to one. Three FMs are added, and the most decorrelated of them sits at maximum rank
correlation **0.9847** against the pack. That is the most decorrelated *non-`lookup`* addition
to this library, but it does not beat `lookup` itself, which sits at **0.9804** on the same
measurement and was the library's most valuable single addition. `lookup` is both stronger and
more decorrelated, which is why it earned +0.000109 and this family earns +0.000006.

**2. The two meta designs are mixed rather than chosen between.** Everyone building on this
library fits either a plain logistic stack over all member logits, or riponce's
missingness-regime design, and then picks the better one. They are not substitutes. Their
percentile-rank mixture beats both, on every partition tested, and the mixing weight sits in
a flat plateau so it is not a fitted parameter.

| meta design | members | OOF AUC | vs. published anchor | folds positive |
|---|---:|---:|---:|---:|
| global logistic stack | 74 | 0.969667 | −0.000026 | 1/15 |
| global logistic stack | 77 | 0.969674 | −0.000020 | 1/15 |
| regime design (published anchor, LB 0.97084) | 74 | 0.969694 | — | — |
| mixture, 2/3 regime + 1/3 global | 74 | 0.969701 | +0.000007 | 14/15 |
| mixture, 2/3 regime + 1/3 global | 74 / 77 | 0.969705 | +0.000011 | 15/15 |
| **mixture + two band-local corrections** | **74 / 77** | **0.969721** | **+0.000027** | **15/15** |

**3. Band-local factorization machines.** The blend is far weaker on some rows than others, and
a model trained *only* on a weak band -- where the class ratio and the value vocabulary both
differ from the global ones -- adds information the 77 global members do not have. Two bands pay;
two others measured negative and are excluded. There is a clean rule for which is which, below.

Every number above is the mean over meta-CV partitions with seeds 42, 2026 and 314159, on
the frozen member split. The leaderboard was not used to choose anything.

## What is being claimed, and what is not

A single public score has a bootstrap standard deviation of about 0.00027 on this test size,
and the *difference* between two submissions correlated at 0.99 has about 0.00007. So a
change of +0.000011 is **below what the leaderboard can resolve**. The bootstrap arithmetic
is szymonkapiski's and it is the reason this notebook reports out-of-fold numbers over three
partitions with per-fold signs, rather than a leaderboard delta. Anyone who tells you they
resolved 1e-5 on 296,302 rows of public test is reading noise.

What is claimed: on 691,369 rows of out-of-fold data, mixing the two designs is better than
picking one, the direction is the same on every partition and in every fold, and it costs no
new models. What is not claimed: that you will see it on the public leaderboard.

## Setup

In [1]:
import glob
import os
import time

import numpy as np
import pandas as pd
from scipy.special import logit
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

TARGET, ID = "addicted_label", "id"
NUM = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
       "work_study_hours", "sleep_hours", "notifications_per_day",
       "app_opens_per_day", "weekend_screen_time"]
CAT = ["gender", "stress_level", "academic_work_impact"]
FE = NUM + CAT
COMP = ["social_media_hours", "gaming_hours", "work_study_hours"]
DAILY = "daily_screen_time_hours"

N_SPLITS, SEED, EPS = 5, 42, 1e-6
NEW = ["fmdeep", "fmpure", "fmwide"]
W_GLOBAL = 1.0 / 3.0          # fixed, not tuned; see the weight curve below
W_BAND = 0.05                 # fixed on band-internal AUC, before any global number
BANDS = {"3-6h": ("bandfm2", 3.0, 6.0), "6-7.8h": ("band_mid", 6.0, 7.8)}
C_GLOBAL, C_REGIME, TOL = 1.0, 0.03, 1e-5


# Kaggle nests /kaggle/input differently depending on how sources are attached, so glob for
# the inputs rather than hardcoding paths.
def find_dir(pattern, *fallbacks):
    for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True):
        return os.path.dirname(p)
    for f in fallbacks:
        if os.path.exists(os.path.join(f, os.path.basename(pattern))):
            return f
    raise FileNotFoundError(pattern)


COMPETITION = find_dir("train.csv", "../data", "data")
LIBRARY = find_dir("oof_lookup.npy", "../data/oof_lib74/oof")
FM_DIR = find_dir("oof_fmdeep.npy", "../outputs/aug4")
print("competition:", COMPETITION)
print("OOF library:", LIBRARY)
print("FM members: ", FM_DIR)

train = pd.read_csv(f"{COMPETITION}/train.csv")
test = pd.read_csv(f"{COMPETITION}/test.csv")
sample = pd.read_csv(f"{COMPETITION}/sample_submission.csv")
y = train[TARGET].to_numpy(np.int8)
n_train, n_test = len(train), len(test)
miss_train = train[FE].isna().sum(axis=1).to_numpy(np.int8)
miss_test = test[FE].isna().sum(axis=1).to_numpy(np.int8)
print(f"{n_train:,} train rows, {n_test:,} test rows, base rate {y.mean():.4f}")

competition: /kaggle/input/competitions/playground-series-s6e8
OOF library: /kaggle/input/s6e8-oof-library-47-models/oof
FM members:  /kaggle/input/s6e8-fm-lattice-blend-members
691,369 train rows, 296,302 test rows, base rate 0.7094


## Why a factorization machine

Two published findings about this dataset point at the same structure.

**The exact value is a lookup key.** tamerlanomralinov measured that `notifications_per_day`
has univariate AUC 0.492, no monotone signal at all, and yet its per-value residuals correlate
0.72 across two independent halves of the data. The generator memorised value→label
associations from the small original dataset, so a column is a lattice of a few hundred to a
few thousand repeated values rather than a continuous magnitude.

**Pairs of columns carry more than single columns.** szymonkapiski measured that target
encoding all 36 numeric pairs at lattice resolution, instead of four hand-picked ones, moved
LightGBM, XGBoost and CatBoost in the same direction, and that going finer still helped
XGBoost.

Pair target encoding estimates each joint cell **independently**. That is where it runs out of
data: `daily_screen_time_hours` has 1,226 distinct values above the rare-value threshold and
`weekend_screen_time` has 1,304, so their joint lattice has on the order of 1.6 million cells
and there are 691,369 rows to fill them. Most cells are empty and the occupied ones are thin.

A factorization machine is the low-rank version of the same object:

$$\hat{y} = b + \sum_f w_{f, x_f} + \sum_{f < g} \langle v_{f, x_f},\, v_{g, x_g} \rangle$$

Each (field, value) gets **one** vector, and the score of a joint cell is an inner product, so
all field pairs are estimated jointly and a thin cell borrows strength from every other cell
that shares one of its values. That is a bilinear function class. The library does not have
one, and the point of adding it is decorrelation rather than raw strength.

## Fields

One integer code per field. Index 0 is missing, and it gets a **learned vector** rather than
an imputed value — the library's own experiments found imputation worth essentially nothing
here, and a learned per-field "missing" vector is the cheaper way to say the same thing.
Index 1 is a shared bucket for values below the count threshold: a cell seen four times cannot
support its own 8-dimensional vector, so it shares one with the other thin cells of its field.
That bucket is the FM's substitute for the `CT_` count column the tree models get.

Nothing here touches the target. Vocabularies and counts are built on train and test together,
which is the same treatment the library's constrained-imputation block gets.

In [2]:
def build_fields(both, min_count=15, coarse=False, derived=False):
    '''Integer code per field, with rare values bucketed. Target-free.'''
    cols = {c: both[c].astype(str).where(both[c].notna(), None) for c in FE}
    if coarse:
        # A denser fallback resolution: the exact cell carries the memorised lookup, and the
        # 0.1-resolution cell still has rows when the exact one does not.
        for c in NUM:
            v = pd.to_numeric(both[c], errors="coerce")
            cols[f"{c}_r1"] = v.round(1).astype(str).where(v.notna(), None)
    if derived:
        # The generator identity daily >= social + gaming + work makes the remainder a real
        # latent variable, so it is worth its own lattice field.
        sgw = both[COMP].sum(axis=1, skipna=False)
        other = both[DAILY] - sgw
        frac = other / both[DAILY].clip(lower=0.1)
        for nm, v in [("other_screen", other.round(2)), ("sgw", sgw.round(2)),
                      ("other_frac", (frac * 50).round() / 50),
                      ("wk_other", (both["weekend_screen_time"] - other).round(1)),
                      ("notif_open", ((both["notifications_per_day"]
                                       / (both["app_opens_per_day"] + 1.0)) * 20).round() / 20)]:
            cols[nm] = v.astype(str).where(v.notna(), None)
        cols["miss_pattern"] = both[FE].isna().astype(int).astype(str).agg("".join, axis=1)

    codes, vocab, field_names = [], [], []
    for name, s in cols.items():
        counts = s.value_counts()
        keep = counts[counts >= min_count].index
        mapping = {v: i + 2 for i, v in enumerate(keep)}   # 0 = missing, 1 = rare bucket
        code = s.map(mapping)
        code = code.where(s.isna() | code.notna(), 1.0)
        codes.append(code.fillna(0).to_numpy(np.int64))
        vocab.append(len(keep) + 2)
        field_names.append(name)
    return np.stack(codes, 1), np.asarray(vocab), field_names


both = pd.concat([train[FE], test[FE]], ignore_index=True)
_, vocab_demo, names_demo = build_fields(both)
print("lattice size per field, exact values above the count threshold:")
for nm, v in zip(names_demo, vocab_demo):
    print(f"  {nm:26s} {v - 2:>5,}")
print(f"\n{len(names_demo)} fields, {int(vocab_demo.sum()):,} embedding rows, "
      f"{len(names_demo) * (len(names_demo) - 1) // 2} field pairs estimated jointly")

lattice size per field, exact values above the count threshold:
  age                           18
  daily_screen_time_hours    1,226
  social_media_hours           655
  gaming_hours                 399
  work_study_hours             595
  sleep_hours                  451
  notifications_per_day        231
  app_opens_per_day            166
  weekend_screen_time        1,304
  gender                         3
  stress_level                   3
  academic_work_impact           2

12 fields, 5,077 embedding rows, 66 field pairs estimated jointly


## The model

The interaction term uses the standard identity

$$\sum_{f<g} \langle v_f, v_g \rangle = \tfrac{1}{2}\Big( \big\|\textstyle\sum_f v_f\big\|^2 - \sum_f \|v_f\|^2 \Big)$$

so it costs one pass over the fields rather than one pass per pair. The optional deep head
over the concatenated embeddings is what takes the model from 0.962 to 0.966; the plain
bilinear form is kept as a separate member because it is the more decorrelated of the two.

In [3]:
import torch
import torch.nn as nn


class FM(nn.Module):
    def __init__(self, total_vocab, n_fields, k=8, deep=384, drop=0.1):
        super().__init__()
        self.v = nn.Embedding(total_vocab, k)      # the lattice vectors
        self.w = nn.Embedding(total_vocab, 1)      # per-value linear weight
        nn.init.normal_(self.v.weight, std=0.01)
        nn.init.zeros_(self.w.weight)
        self.bias = nn.Parameter(torch.zeros(1))
        self.deep = nn.Sequential(
            nn.Linear(n_fields * k, deep), nn.GELU(), nn.Dropout(drop),
            nn.Linear(deep, deep // 2), nn.GELU(), nn.Dropout(drop),
            nn.Linear(deep // 2, 1),
        ) if deep else None

    def forward(self, idx):
        e = self.v(idx)                            # (batch, fields, k)
        s = e.sum(1)
        interaction = 0.5 * (s.pow(2).sum(1) - e.pow(2).sum((1, 2)))
        out = self.bias + self.w(idx).sum((1, 2)) + interaction
        if self.deep is not None:
            out = out + self.deep(e.flatten(1)).squeeze(-1)
        return out


print(FM(10_000, 12))

FM(
  (v): Embedding(10000, 8)
  (w): Embedding(10000, 1)
  (deep): Sequential(
    (0): Linear(in_features=96, out_features=384, bias=True)
    (1): GELU(approximate='none')
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=384, out_features=192, bias=True)
    (4): GELU(approximate='none')
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=192, out_features=1, bias=True)
  )
)


## Training

`RUN_TRAINING = False` loads the three members from the
[published arrays](https://www.kaggle.com/datasets/raykkretzschmar/s6e8-fm-lattice-blend-members),
which is a few seconds and reproduces the exact numbers below. `True` trains them, which is
about ten minutes per member on a GPU and about forty on CPU. The training code below runs
either way; the switch only decides whether this notebook executes it now.

Three details earn their place. Embedding weight decay is set an order of magnitude above the
rest of the network, because the lattice vectors are the part that overfits. Random extra
masking hides values during training, so the model sees far more missingness patterns than the
data happens to contain. And an exponential moving average of the weights is what gets
evaluated and kept, which on a model this small is worth more than any single learning rate.

In [4]:
RUN_TRAINING = False

CONFIGS = {
    "fmdeep": dict(k=8, deep=384, aug=0.20, coarse=False, derived=False, seeds=3),
    "fmpure": dict(k=16, deep=0, aug=0.10, coarse=True, derived=True, seeds=3),
    "fmwide": dict(k=32, deep=256, aug=0.15, coarse=True, derived=True, seeds=2),
}
FOLDS = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
             .split(np.zeros(n_train), y))


def train_member(cfg, epochs=100, patience=8, bs=4096, lr=3e-3, emb_wd=3e-4, min_count=15):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    codes, vocab, field_names = build_fields(both, min_count, cfg["coarse"], cfg["derived"])
    offset = np.concatenate([[0], np.cumsum(vocab)[:-1]])
    IDX = torch.from_numpy(codes + offset[None, :])
    OFFSET = torch.from_numpy(offset).to(device)
    Y = torch.from_numpy(y.astype(np.float32))
    total = int(vocab.sum())
    oof, test_pred = np.zeros(n_train), np.zeros(n_test)
    Ite = IDX[n_train:].to(device)
    t0 = time.time()

    for fold, (itr, iva) in enumerate(FOLDS):
        fold_oof, fold_test = np.zeros(len(iva)), np.zeros(n_test)
        for s in range(cfg["seeds"]):
            torch.manual_seed(1000 * s + fold)
            model = FM(total, len(field_names), cfg["k"], cfg["deep"]).to(device)
            emb = [model.v.weight, model.w.weight]
            rest = [p for nm, p in model.named_parameters() if not nm.startswith(("v.", "w."))]
            opt = torch.optim.AdamW([{"params": rest, "weight_decay": 1e-5},
                                     {"params": emb, "weight_decay": emb_wd}], lr=lr)
            steps = -(-len(itr) // bs) * epochs + 10
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps,
                                                        pct_start=0.2)
            lossf = nn.BCEWithLogitsLoss()
            params = list(model.parameters())
            ema = [p.detach().clone() for p in params]
            Itr, Ytr, Iva = IDX[itr].to(device), Y[itr].to(device), IDX[iva].to(device)
            gen = torch.Generator().manual_seed(7 * s + fold)

            def predict(t, chunk=65536):
                model.eval()
                with torch.no_grad():
                    return torch.cat([model(t[i:i + chunk])
                                      for i in range(0, len(t), chunk)]).float().cpu().numpy()

            best, best_w, bad = 0.0, None, 0
            for ep in range(epochs):
                model.train()
                perm = torch.randperm(len(itr), generator=gen).to(device)
                for i in range(0, len(itr), bs):
                    sl = perm[i:i + bs]
                    idx = Itr[sl]
                    if cfg["aug"] > 0:
                        hide = torch.rand(idx.shape, device=device) < cfg["aug"]
                        idx = torch.where(hide, OFFSET.expand_as(idx), idx)
                    loss = lossf(model(idx), Ytr[sl])
                    opt.zero_grad(set_to_none=True)
                    loss.backward()
                    nn.utils.clip_grad_norm_(params, 2.0)
                    opt.step()
                    sched.step()
                    with torch.no_grad():
                        torch._foreach_mul_(ema, 0.999)
                        torch._foreach_add_(ema, [p.detach() for p in params], alpha=0.001)
                if ep >= 2:
                    backup = [p.detach().clone() for p in params]
                    with torch.no_grad():
                        for p, e in zip(params, ema):
                            p.copy_(e)
                    auc = roc_auc_score(y[iva], predict(Iva))
                    if auc > best:
                        best, best_w, bad = auc, [e.detach().clone() for e in ema], 0
                    else:
                        bad += 1
                    with torch.no_grad():
                        for p, b in zip(params, backup):
                            p.copy_(b)
                    if bad >= patience:
                        break
            with torch.no_grad():
                for p, e in zip(params, best_w):
                    p.copy_(e)
            fold_oof += predict(Iva) / cfg["seeds"]
            fold_test += predict(Ite) / cfg["seeds"]
        oof[iva] = fold_oof
        test_pred += fold_test / N_SPLITS
        print(f"    fold {fold} AUC {roc_auc_score(y[iva], oof[iva]):.5f} "
              f"({time.time() - t0:.0f}s)", flush=True)
    return oof, test_pred


FM_OOF, FM_TEST = {}, {}
for name, cfg in CONFIGS.items():
    if RUN_TRAINING:
        print(f"{name}: {cfg}", flush=True)
        FM_OOF[name], FM_TEST[name] = train_member(cfg)
        np.save(f"oof_{name}.npy", FM_OOF[name])
        np.save(f"test_{name}.npy", FM_TEST[name])
    else:
        FM_OOF[name] = np.load(f"{FM_DIR}/oof_{name}.npy").astype(np.float64)
        FM_TEST[name] = np.load(f"{FM_DIR}/test_{name}.npy").astype(np.float64)
    assert FM_OOF[name].shape == (n_train,) and FM_TEST[name].shape == (n_test,)
    print(f"  {name:8s} OOF AUC {roc_auc_score(y, FM_OOF[name]):.5f}")

  fmdeep   OOF AUC 0.96666
  fmpure   OOF AUC 0.96455
  fmwide   OOF AUC 0.96493


## Load the 74-member library

Every array in it is on the same frozen split, `StratifiedKFold(5, shuffle=True,
random_state=42)`, which is what makes stacking other people's out-of-fold predictions honest.
Check that before reusing anyone's OOF, including these: most published S6E8 OOF arrays use a
different fold count or average over seeds, and they all look perfectly healthy while quietly
inflating a blend built on this split.

In [5]:
names = sorted(os.path.basename(p)[4:-4] for p in glob.glob(f"{LIBRARY}/oof_*.npy"))
O, T = [], []
for nm in names:
    o = np.load(f"{LIBRARY}/oof_{nm}.npy").astype(np.float64)
    t = np.load(f"{LIBRARY}/test_{nm}.npy").astype(np.float64)
    assert o.shape == (n_train,) and t.shape == (n_test,), nm
    O.append(o)
    T.append(t)
O74, T74 = np.column_stack(O), np.column_stack(T)
print(f"{len(names)} library members")

# float64 throughout is deliberate. Members agree to about 0.99, so the meta model works on
# differences finer than float32 resolution near 0 and 1; downcasting flips test-row ranks.
OL74 = logit(np.clip(O74, EPS, 1 - EPS))
TL74 = logit(np.clip(T74, EPS, 1 - EPS))
OL77 = np.column_stack([OL74] + [logit(np.clip(FM_OOF[n], EPS, 1 - EPS)) for n in NEW])
TL77 = np.column_stack([TL74] + [logit(np.clip(FM_TEST[n], EPS, 1 - EPS)) for n in NEW])
print("global design:", OL77.shape, "| regime design is built from:", OL74.shape)

74 library members
global design: (691369, 77) | regime design is built from: (691369, 74)


## What the new members are worth

Strength is the wrong statistic to judge a new member by. `lat_xgb` scores 0.96749 and
contributes exactly nothing, because it correlates 0.998 with something already present;
`pubmk_nn` scores 0.94085 and earns a weight. The number that matters is **maximum rank
correlation against the existing pack** — but it is not sufficient on its own, and the table
below is the evidence. Every figure in it is computed here, on all 691,369 out-of-fold rows,
so the reference row for `lookup` is directly comparable rather than quoted from elsewhere.

In [6]:
pack_ranks = np.column_stack([rankdata(O74[:, i]) for i in range(O74.shape[1])])
rows = []
for nm in NEW:
    r = rankdata(FM_OOF[nm])
    cors = np.array([np.corrcoef(r, pack_ranks[:, i])[0, 1] for i in range(pack_ranks.shape[1])])
    rows.append({"member": nm, "OOF AUC": roc_auc_score(y, FM_OOF[nm]),
                 "max rank corr": cors.max(), "closest": names[int(cors.argmax())]})

lookup_r = rankdata(O74[:, names.index("lookup")])
lookup_cors = [np.corrcoef(lookup_r, pack_ranks[:, i])[0, 1]
               for i in range(pack_ranks.shape[1]) if names[i] != "lookup"]
rows.append({"member": "lookup (reference)", "OOF AUC": roc_auc_score(y, O74[:, names.index("lookup")]),
             "max rank corr": max(lookup_cors), "closest": "-"})
del pack_ranks
pd.DataFrame(rows).style.format({"OOF AUC": "{:.5f}", "max rank corr": "{:.4f}"}).hide(axis="index")

member,OOF AUC,max rank corr,closest
fmdeep,0.96666,0.9888,pub_tabm
fmpure,0.96455,0.9847,naji04
fmwide,0.96493,0.9877,naji04
lookup (reference),0.96853,0.9804,-


## The two meta designs

**Global.** One logistic regression over all member logits, `C=1.0`. Several weights come out
negative, which is not a bug: with members correlated above 0.99 the meta model uses some of
them as corrections rather than as independent opinions.

**Regime.** riponce's design, ported into the library's own `src/blend_regime.py`: the same
logits, plus each logit multiplied by *is this row complete*, by *is it missing four or more
fields*, and by a normalised measure of how much the members disagree, plus five aggregates.
It is better than global by about +0.00003 because accuracy collapses as fields go missing and
the pack should be trusted differently there.

**Why the new members go into the global design only.** The regime design's width is
`4M + 5`. Every member added widens it by four columns, so adding members to it changes the
information *and* the effective regularisation at the same time — and the second effect wins.
The three FMs are worth +0.000006 in the global design (z ≈ +5 on all three partitions) and
−0.000025 in the regime design, and re-tuning `C` down to 0.01 or 0.003 makes the regime
design worse still, not better. So each design is given the member set that measured best for
it, which is one binary choice supported by three partitions rather than a search.

In [7]:
def regime_design(lg, missing, dmean=None, dstd=None):
    '''riponce's design: global logits plus complete / severe / disagreement interactions.'''
    complete = (missing == 0).astype(np.float64)[:, None]
    severe = (missing >= 4).astype(np.float64)[:, None]
    d = lg.std(axis=1, keepdims=True)
    if dmean is None:
        dmean, dstd = float(d.mean()), float(d.std())
    dn = (d - dmean) / (dstd + 1e-6)
    agg = np.column_stack([lg.mean(1), lg.std(1), lg.max(1) - lg.min(1),
                           complete[:, 0], severe[:, 0]])
    return np.column_stack([lg, lg * complete, lg * severe, lg * dn, agg]), dmean, dstd


def honest(X, C, seed=SEED):
    '''Refit the meta model inside each fold; it only ever predicts rows it did not see.

    StandardScaler is not cosmetic here: lbfgs will not converge on the raw regime design at
    any sane max_iter, and a non-converged fit reads higher than the truth.
    '''
    pred = np.zeros(n_train)
    fold_id = np.zeros(n_train, np.int8)
    for f, (itr, iva) in enumerate(StratifiedKFold(N_SPLITS, shuffle=True,
                                                   random_state=seed).split(X, y)):
        sc = StandardScaler().fit(X[itr])
        m = LogisticRegression(C=C, max_iter=5000, solver="lbfgs", tol=TOL)
        m.fit(sc.transform(X[itr]), y[itr])
        assert int(np.max(m.n_iter_)) < 5000, "meta model did not converge"
        pred[iva] = m.predict_proba(sc.transform(X[iva]))[:, 1]
        fold_id[iva] = f
    return pred, fold_id


t0 = time.time()
MR74, dmean, dstd = regime_design(OL74, miss_train)
print(f"regime design width {MR74.shape[1]}, global design width {OL77.shape[1]}")

oof_regime, fold_id = honest(MR74, C_REGIME)
print(f"regime74  OOF {roc_auc_score(y, oof_regime):.6f}  ({time.time() - t0:.0f}s)", flush=True)
oof_global, _ = honest(OL77, C_GLOBAL)
print(f"global77  OOF {roc_auc_score(y, oof_global):.6f}  ({time.time() - t0:.0f}s)", flush=True)

regime design width 301, global design width 77
regime74  OOF 0.969687  (387s)
global77  OOF 0.969665  (630s)


## Mixing beats choosing

Both designs see the same members and optimise the same loss, and they still disagree about
individual rows, because the regime design spends its capacity on where to distrust the pack
while the global one spends it on the pack itself. Percentile-ranking both and averaging them
is better than either.

The weight is fixed at **1/3** on the global side. The measured optimum was 0.30–0.35 on all
three partitions, the curve is flat to 1e-6 across that whole range, and picking the exact
argmax of a flat curve is fitting noise. A round number inside the plateau is not a parameter.

In [8]:
def pct(v):
    return rankdata(v) / len(v)


r_rank, g_rank = pct(oof_regime), pct(oof_global)
curve = pd.DataFrame(
    [{"weight on global": w, "OOF AUC": roc_auc_score(y, (1 - w) * r_rank + w * g_rank)}
     for w in np.arange(0.0, 0.71, 0.05)])
mix_oof = (1 - W_GLOBAL) * r_rank + W_GLOBAL * g_rank

base = roc_auc_score(y, oof_regime)
print(f"regime74 anchor        {base:.6f}")
print(f"global77 alone         {roc_auc_score(y, oof_global):.6f}")
print(f"mixture at w = 1/3     {roc_auc_score(y, mix_oof):.6f}  "
      f"({roc_auc_score(y, mix_oof) - base:+.6f})")
print("\nper fold, mixture minus anchor:")
for f in range(N_SPLITS):
    m = fold_id == f
    d = roc_auc_score(y[m], mix_oof[m]) - roc_auc_score(y[m], r_rank[m])
    print(f"  fold {f}  {d:+.8f}")
curve.style.format({"weight on global": "{:.2f}", "OOF AUC": "{:.6f}"}).hide(axis="index")

regime74 anchor        0.969687
global77 alone         0.969665
mixture at w = 1/3     0.969697  (+0.000009)

per fold, mixture minus anchor:
  fold 0  +0.00001523
  fold 1  +0.00001182
  fold 2  +0.00000341
  fold 3  +0.00000913
  fold 4  +0.00001000


weight on global,OOF AUC
0.00,0.969687
0.05,0.969690
0.10,0.969692
0.15,0.969694
0.20,0.969695
0.25,0.969696
0.30,0.969696
0.35,0.969697
0.40,0.969696
0.45,0.969696


## Repeated on two more meta partitions

The fold assignment above is seed 42, which is also the split the members themselves were
trained on. A difference this small has to survive a change of meta partition, so the whole
comparison was re-run with meta-CV seeds 2026 and 314159. Those runs are stored rather than
executed here, because each is another pair of blend fits and this notebook already spends
most of its time on the first one.

| meta design | members | seed 42 | seed 2026 | seed 314159 | mean | folds positive |
|---|---:|---:|---:|---:|---:|---:|
| global | 74 | 0.969658 | 0.969667 | 0.969676 | 0.969667 | 1/15 |
| global | 77 | 0.969665 | 0.969674 | 0.969683 | 0.969674 | 1/15 |
| regime (anchor) | 74 | 0.969688 | 0.969690 | 0.969704 | 0.969694 | — |
| mixture | 74/74 | 0.969693 | 0.969698 | 0.969713 | 0.969701 | 14/15 |
| mixture | 74/77 | 0.969697 | 0.969701 | 0.969716 | 0.969705 | 15/15 |
| **mixture + 2 bands** | **74/77** | **0.969713** | **0.969717** | **0.969732** | **0.969721** | **15/15** |

The anchor moves by 0.000016 across partitions and the mixture's gain over it is +0.000009,
+0.000011, +0.000012 — smaller than the partition-to-partition wobble in absolute terms, which
is exactly why it is measured *paired* on the same rows, where it is the same sign in all
fifteen folds.

## Where the blend is weak, and what to do about it

Accuracy is wildly uneven across `daily_screen_time`. The high bands are easy because almost
every row in them is positive, not because they are modelled well; the middle is where the
genuinely uncertain rows live. This diagnosis is szymonkapiski's, from `src/train_band.py` in the
OOF library, and it comes with a warning: giving the **meta model** its own weights inside a band
changed nothing (0.96940 to 0.96939), and a band-specialised **gradient boosting tree** was
written and never shipped. His conclusion was that the members do not contain the information.

They do, but only a model of the right *class* trained on the right *rows* can extract it. Two
things have to line up at once:

- the signal here is a value lattice, so the model has to read exact values -- a tree on the band
  has the right training set and the wrong function class;
- global training spends its lattice vectors on 691,369 rows of which 43% are trivially positive,
  so a globally-trained lattice model has the right class and the wrong training set.

A factorization machine fitted **only on the band**, with its vocabulary rebuilt from band rows,
has both. And there is a sharp rule for when it pays:

| subpopulation | rows | pos rate | blend AUC there | band FM | gap | band gain |
|---|---:|---:|---:|---:|---:|---:|
| `daily <= 3h` | 25,362 | 0.231 | 0.90790 | 0.83543 | 0.0725 | **-0.000070** |
| `daily 3-6h` | 158,451 | 0.341 | 0.91829 | 0.90565 | 0.0126 | +0.000041 |
| `daily 6-7.8h` | 115,842 | 0.643 | 0.93383 | 0.91935 | 0.0145 | **+0.000086** |
| `n_missing >= 4` | 53,700 | 0.711 | 0.93051 | 0.90046 | 0.0301 | **-0.000091** |

What decides the sign is the **gap** between the band model and the blend on those rows -- not the
band's size, not its class balance, not how weak the blend is there. Under about 0.015 it pays;
at 0.03 it costs. A control matters here: a band-only LightGBM reproduces the known negative
(0.90493 in the 3-6h band, best weight 0.00), and an RBF random-Fourier kernel reaches rank
correlation **0.80** with the blend -- by far the most decorrelated prediction in this project --
and is still worth exactly zero, at 0.83083. Decorrelation is necessary and nowhere near enough.

The correction is applied as a **within-band reordering**. Band rows keep their global percentile
positions and only their order among themselves changes, so cross-band interleaving is untouched
and the prediction's value multiset is unchanged -- asserted below. The weight 0.05 was fixed on
band-internal AUC before any global number was computed, so it is not a grid mined against the
metric being reported.

In [9]:
def band_adjust(base, idx, band_pred, w=W_BAND):
    '''Reorder rows within a band, preserving their global percentile positions.

    Only the ORDER inside the band changes. The band's set of predicted values is reused
    exactly, reassigned by the blended ranking, so nothing about how band rows interleave with
    non-band rows can move -- which is what keeps a local improvement from doing global damage.
    '''
    out = base.copy()
    mixed = (1 - w) * (rankdata(base[idx]) / len(idx)) + w * (rankdata(band_pred) / len(idx))
    out[idx] = np.sort(base[idx])[np.argsort(np.argsort(mixed))]
    return out


daily_tr = pd.to_numeric(train[DAILY], errors="coerce").to_numpy()
daily_te = pd.to_numeric(test[DAILY], errors="coerce").to_numpy()

corrected = mix_oof.copy()
for label, (name, lo, hi) in BANDS.items():
    idx = np.flatnonzero((daily_tr > lo) & (daily_tr <= hi))
    band_oof = np.load(f"{FM_DIR}/bandoof_{name}.npy")
    assert band_oof.shape == idx.shape, (label, band_oof.shape, idx.shape)
    before = roc_auc_score(y[idx], corrected[idx])
    corrected = band_adjust(corrected, idx, band_oof)
    print(f"{label:8s} {len(idx):>7,} rows  band FM {roc_auc_score(y[idx], band_oof):.5f}"
          f"  band-internal {before:.5f} -> {roc_auc_score(y[idx], corrected[idx]):.5f}")

print(f"\nmixture            {roc_auc_score(y, mix_oof):.7f}")
print(f"+ band corrections {roc_auc_score(y, corrected):.7f}  "
      f"({roc_auc_score(y, corrected) - roc_auc_score(y, mix_oof):+.7f})")
print("\nper fold, corrected minus mixture:")
for f in range(N_SPLITS):
    m = fold_id == f
    print(f"  fold {f}  {roc_auc_score(y[m], corrected[m]) - roc_auc_score(y[m], mix_oof[m]):+.8f}")

3-6h     158,451 rows  band FM 0.90565  band-internal 0.91829 -> 0.91833
6-7.8h   115,842 rows  band FM 0.91935  band-internal 0.93383 -> 0.93391

mixture            0.9696966
+ band corrections 0.9697125  (+0.0000159)

per fold, corrected minus mixture:
  fold 0  +0.00002649
  fold 1  +0.00002020
  fold 2  +0.00001715
  fold 3  +0.00001197
  fold 4  +0.00000211


## Submission

In [10]:
# Both meta models are refit on all 691,369 training rows for the final prediction, which is
# what every published version of this stack does.
def fit_predict(X, Xt, C):
    sc = StandardScaler().fit(X)
    m = LogisticRegression(C=C, max_iter=5000, solver="lbfgs", tol=TOL).fit(sc.transform(X), y)
    assert int(np.max(m.n_iter_)) < 5000
    return m.predict_proba(sc.transform(Xt))[:, 1]


MRT74, _, _ = regime_design(TL74, miss_test, dmean, dstd)
test_regime = fit_predict(MR74, MRT74, C_REGIME)
test_global = fit_predict(OL77, TL77, C_GLOBAL)
prediction = (1 - W_GLOBAL) * pct(test_regime) + W_GLOBAL * pct(test_global)

before_bands = prediction.copy()
for label, (name, lo, hi) in BANDS.items():
    idx = np.flatnonzero((daily_te > lo) & (daily_te <= hi))
    band_test = np.load(f"{FM_DIR}/bandtest_{name}.npy")
    assert band_test.shape == idx.shape, (label, band_test.shape, idx.shape)
    prediction = band_adjust(prediction, idx, band_test)
    print(f"{label:8s} corrected {len(idx):,} test rows")

# A within-band reordering may not touch the value multiset, and may not move a row outside
# any band. Both are cheap to assert and would catch a mis-indexed band immediately.
assert np.allclose(np.sort(prediction), np.sort(before_bands))
in_any_band = np.zeros(len(test), bool)
for _, lo, hi in BANDS.values():
    in_any_band |= (daily_te > lo) & (daily_te <= hi)
assert not (~np.isclose(prediction, before_bands) & ~in_any_band).any()

submission = pd.DataFrame({ID: test[ID], TARGET: prediction})
assert list(submission.columns) == [ID, TARGET]
assert submission[ID].equals(sample[ID])
assert len(submission) == 296_302
assert np.isfinite(submission[TARGET]).all()
assert submission[TARGET].between(0, 1).all()
assert submission[TARGET].nunique() > 250_000
submission.to_csv("submission_band_model.csv", index=False)
print(f"wrote submission_band_model.csv, {len(submission):,} rows, "
      f"{submission[TARGET].nunique():,} distinct values")
print(f"rank correlation of the two designs on test: "
      f"{np.corrcoef(pct(test_regime), pct(test_global))[0, 1]:.5f}")
submission.head()

3-6h     corrected 70,351 test rows
6-7.8h   corrected 50,917 test rows
wrote submission_band_model.csv, 296,302 rows, 265,510 distinct values
rank correlation of the two designs on test: 0.99796


,id,addicted_label
0,691369,0.795939
1,691370,0.476935
2,691371,0.361683
3,691372,0.608787
4,691373,0.673185


## August 11 audit: more members and a source-data signal that does not transfer

The band model above was the end of the first round of work. Since then, two genuinely new
sources of information became available.

The first is Dariush Afshar's
[94 verified OOF stack](https://www.kaggle.com/code/dariushafshar/94-verified-oofs-honest-cv-0-96985-lb-0-97097).
It starts with this notebook's FM family and adds twelve independently audited level-1 members:
fixed-schedule lookup transformers, exact-value and structural GBDTs, a RealMLP, and a second
lookup seed. Against its 82-member reference, the additions improve all five nested comparison
folds by a mean **+0.000101**. This is the right kind of growth: new representations admitted by
paired OOF evidence, not another blind public blend.

The second is Kodai Fukuda's
[exact-value/ORIG XGBoost](https://www.kaggle.com/code/kodaifukuda0311/s6e8-xgb-the-power-of-exact-value-te).
It target-encodes all twelve raw features by exact repeated value and adds distributional
features from the original 7,500-row dataset. Its five-seed OOF AUC is **0.968268**, and adding it
to the 94-member global arm improved OOF **0.969727 → 0.969795**, positive in 5/5 folds.

That clean OOF result did *not* improve the public board: the 95-member stack scored 0.97094,
against 0.97095 for the 94-member control. This is a useful negative result. At correlations
above 0.999, even a real OOF gain can be smaller than public-split noise. The model stays in the
research record but is not in the final prediction.

The clean conclusion is therefore the 94-member control. It has the strongest fully audited
level-1 provenance in this experiment, improves every paired comparison fold against its
82-member reference, and avoids selecting a model because of a one-point public-LB fluctuation.

| candidate | paired evidence | public LB | decision |
|---|---:|---:|---|
| earlier meta mixture + band FMs | +0.000027, 15/15 folds | 0.97083–0.97084 | useful first-stage result |
| 94 members + exact-value/ORIG XGB | +0.000069, 5/5 folds | 0.97094 | keep as a negative result |
| **audited 94-member stack** | **+0.000101 vs 82 members, 5/5 folds** | **0.97095** | **notebook output** |

A separate 90/10 residual experiment reached 0.97096, but it depends on a prediction dataset
whose licence Kaggle marks as unknown. It is deliberately not a source or output of this public
notebook. The artifact produced below is the cleaner 0.97095 submission.

In [11]:
# Build the exact scored 0.97095 submission. The complete 94-member notebook is attached as a
# kernel source, so its executed submission can be reused without copying predictions into a
# new dataset.
def locate_source(fragment, filename, local_fallback=None):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
            if fragment in p.lower()]
    if hits:
        return sorted(hits)[0]
    if local_fallback and os.path.exists(local_fallback):
        return local_fallback
    raise FileNotFoundError(f"could not find {filename!r} from source {fragment!r}")
anchor_path = locate_source(
    "94-verified-oofs-honest-cv", "submission.csv",
    "../research_aug11/dariush94/output/submission.csv")
anchor = pd.read_csv(anchor_path)
assert list(anchor.columns) == [ID, TARGET]
assert anchor[ID].equals(test[ID])
submission = anchor.copy()
assert submission[ID].equals(sample[ID])
assert submission.shape == (296_302, 2)
assert np.isfinite(submission[TARGET]).all()
assert submission[TARGET].between(0, 1).all()
assert submission[TARGET].nunique() == len(submission)
submission.to_csv("submission.csv", index=False)
print("94-member source:", anchor_path)
print(f"wrote submission.csv: {len(submission):,} rows, "
      f"{submission[TARGET].nunique():,} unique predictions")
submission.head()

94-member source: /kaggle/input/94-verified-oofs-honest-cv-0-96985-lb-0-97097/submission.csv
wrote submission.csv: 296,302 rows, 296,302 unique predictions


,id,addicted_label
0,691369,0.797877
1,691370,0.470123
2,691371,0.353526
3,691372,0.616005
4,691373,0.690662


## August 13 update: a three-source rank blend (public LB 0.97099)

The OOF work above says which additions are defensible before looking at the leaderboard. It
does not say that the final few ten-thousandths can be estimated reliably from one public
split. The following blend is therefore reported for what it is: a public-LB-informed ensemble.

[Daniil Krasnov](https://www.kaggle.com/code/daniilkrasnovvv/s6e8-top-1-public-0-97099)
noticed that three public predictions still disagree enough to benefit from rank averaging:

- this notebook's audited 94-member stack;
- [Naji's 0.97097 stack](https://www.kaggle.com/code/najiama/s6e8-addiction-lb-0-97097);
- [Nikita's smartphone-addiction model](https://www.kaggle.com/code/nikita7364777/smartphone-addiction-notebook).

Each prediction is converted to a percentile rank before averaging. That removes calibration
differences and makes the contribution of each source exactly one third. No row-wise rules,
band weights or hand-tuned coefficients are fitted in this step. The exact file produced below
scored **0.97099** on the public leaderboard.


In [12]:
# The source notebook executes the three rank transforms and their equal-weight average. It is
# attached directly because Kaggle does not transitively mount that notebook's two own sources
# when this notebook is executed. Reading its output preserves the exact scored floating-point file.
blend_path = locate_source("s6e8-top-1-public-0-97099", "submission.csv")
submission = pd.read_csv(blend_path)
assert list(submission.columns) == [ID, TARGET]
assert submission[ID].equals(sample[ID])
assert submission.shape == (296_302, 2)
assert np.isfinite(submission[TARGET]).all()
assert submission[TARGET].between(0, 1).all()
submission.to_csv("submission.csv", index=False)

print("three-way rank blend source:", blend_path)
print(f"wrote submission.csv: {len(submission):,} rows")
print("verified public LB: 0.97099")
submission.head()


three-way rank blend source: /kaggle/input/s6e8-top-1-public-0-97099/submission.csv
wrote submission.csv: 296,302 rows
verified public LB: 0.97099


,id,addicted_label
0,691369,0.806278
1,691370,0.475567
2,691371,0.359301
3,691372,0.609352
4,691373,0.691040


## Negative results

Measured on the same folds, so these are dead ends rather than untested guesses. Several of
them are the obvious next thing to try, which is why they are here.

| idea | result |
|---|---|
| LightGBM as the meta model on the 74 member logits | 0.969547, worse than logistic by 0.00011 |
| LightGBM meta, plus the raw columns and missingness pattern | 0.969508 |
| logistic meta, plus missingness indicator columns | 0.969655, indistinguishable from plain |
| the new members added to the **regime** design, `C` re-tuned to 0.01 / 0.003 | 0.969654 / 0.969596, both worse than leaving them out |
| a regime design whose interaction block does not grow with `M` (aggregates + top-K only) | best 0.969680 at K=16, against 0.969694 for the full design; closer than it looks but still behind |
| adding the LightGBM meta as a third component of the mixture at 5–15% | +0.000001, noise |
| cross-network (DCN-style) crosses on top of the FM embeddings | 0.96512–0.96553 on fold 0, against 0.96579 without them |
| coarse and derived lattice fields in the **deep** FM | 0.96493 against 0.96579 without them (they help the *pure* FM) |
| a second-stage residual putting the FM family on top of the regime blend alone | +0.000003, 14/15 folds — real, but the mixture subsumes it |
| making the *global* FM members stronger (0.96666 → 0.96739 solo, via the smooth branch) | +0.000000 in the mixture. Once a function class is present, strengthening it does nothing |
| retraining the Lookup-Transformer with a different readout, depth and width | 0.9926 correlation with the library's existing `lookup` — a twin, despite the architecture change |
| adding a pairwise AUC surrogate to the FM loss | 0.9965 correlation with its own BCE counterpart. The objective is not a diversity axis |
| per-regime cross-fitted isotonic calibration | −0.000080, 0/15 folds. Regime base rates are 0.7081 / 0.7102 / 0.7106, so there is no miscalibration to exploit |
| bagging the global meta over random 60% member subsets | 0.96960 against 0.969665 for the full fit. With 691k rows the meta is not variance-limited |
| band-local FM on `daily <= 3h` and on `n_missing >= 4` | −0.000070 and −0.000091: the gap to the blend is too large (0.072 and 0.030) |

The first three are worth dwelling on: meta-model *capacity* is not where the headroom is on
this problem. Three different attempts to give the meta model more expressive power all lost
to plain logistic regression, and the thing that actually worked was averaging two
differently-biased linear metas. When 74 members agree to 0.99, the meta model's job is
variance reduction, and a flexible model does that worse.

## Credits

- **szymonkapiski** — the [74-model OOF library](https://www.kaggle.com/datasets/szymonkapiski/s6e8-oof-library-47-models)
  and the frozen split that makes it stackable, the lattice target-encoding family, the
  constrained-imputation block, and the bootstrap arithmetic for what the leaderboard can
  resolve. This notebook is built on that library and would not exist without it.
- **riponce** — the missingness-regime interaction design, which is the stronger of the two
  metas mixed here.
- **tamerlanomralinov** — the Lookup-Transformer and the split-half residual test showing that
  exact values are lookup keys, which is the argument for using a factorization machine at all.
- **omidbaghchehsaraei** — target encoding every column including the numerics; **najiama** —
  five level-1 members in the public OOF library;
  **dariushafshar** — the audited 94-member pool and its twelve verified additions;
  **kodaifukuda0311** — the exact-value/ORIG XGBoost and the external-reference feature family.
- **nikita7364777** — the independent smartphone-addiction prediction used in the final rank blend;
  **daniilkrasnovvv** — identifying and publishing the equal-weight three-source combination.
